1. Cấu hình — chỉ dữ liệu mô phỏng

Run All offline. Không lấy CSV thật, gọi API, cài package hoặc chạy thuật toán eligibility. UUID của fixtures có namespace/key riêng; mọi nguồn/quán/giá/reviewer đều mô phỏng.

In [ ]:
from pathlib import Path
import sys, platform, json
import pandas as pd
here = Path.cwd().resolve()
DATA_PREPARATION_DIR = next((p if p.name == 'data-preparation' else p / 'data-preparation' for p in [here, *here.parents] if (p if p.name == 'data-preparation' else p / 'data-preparation').is_dir()), None)
if DATA_PREPARATION_DIR is None:
    raise RuntimeError('Mở kernel trong repo hoặc đặt DATA_PREPARATION_DIR')
sys.path.insert(0, str(DATA_PREPARATION_DIR / 'scripts'))
import food_fixtures as ff
OPTIONS = globals().get('NOTEBOOK_OPTIONS', {})
if OPTIONS.get('refresh') or OPTIONS.get('allow_api'):
    raise ValueError('Notebook fixtures không dùng refresh/API')
OUTPUT_DIR = DATA_PREPARATION_DIR.parent / 'tests/fixtures/food-data-v1'
REPORTS_DIR = DATA_PREPARATION_DIR / 'datasets/food-fixtures/reports'
print({'python': platform.python_version(), 'pandas': pd.__version__, 'output': str(OUTPUT_DIR), 'fixtureOnly': True, 'part3Commit': ff.PART3_COMMIT})


2. Khám phá dữ liệu cơ sở

10 món ứng viên để kiểm trần 8. Tọa độ là ô lưới giả lập, không phải GPS người dùng hay dữ liệu quán thật. CSV/JSON theo contract 1.0.0; version dataset 1.0.0 chỉ dành cho fixtures.

In [ ]:
datasets, suite, registry = ff.build_suite()
base = datasets['base']
print('Số dòng từng entity:', {k: len(v) for k, v in base['entities'].items()})
for entity in ['taxonomy', 'dishes', 'venues', 'venueDishes', 'coverageAreas', 'publicAnchors', 'dataSources']:
    print(entity, 'columns=', list(ff.dc.ENTITIES[entity]['fields']))
    display(pd.DataFrame(base['entities'][entity]))
print('IDs/key registry:', len(registry))
display(pd.DataFrame(registry))


3. Ma trận ca và đầu vào

expected được viết rõ trong module, không lấy từ một thuật toán lọc/xếp hạng. Case chỉ dẫn spec và dataset đầy đủ; thứ tự xếp hạng cụ thể chưa chốt. A chọn nướng, B chọn lẩu là ưu tiên mềm.

In [ ]:
print('validationAt:', suite['validationAt'], 'cases:', len(suite['cases']), 'datasets:', len(datasets))
case_table = pd.DataFrame([dict(id=c['id'], group=c['group'], dataset=c['datasetKey'], evaluatedAt=c['evaluatedAt'], desiredAt=c['input']['context']['desiredAt'], description=c['description']) for c in suite['cases']])
display(case_table)
display(case_table['group'].value_counts().rename('cases'))
first = suite['cases'][0]
print(json.dumps(first['input'], ensure_ascii=False, indent=2))


4. Kết quả kỳ vọng và phép so sánh

Eligible/excluded/needsConfirmation partition từng offering. Pool chống trùng theo (dishId, variant); hai quán bán cùng món/variant không thành hai lựa chọn. Nếu hơn 8 ứng viên thì kiểm kích thước và đại diện nhóm, không chỉ định tám món thắng. So sánh cùng seed/context kiểm cùng ordered pool của thuật toán tương lai, không tính thứ tự ở đây.

In [ ]:
for c in suite['cases']:
    print(c['id'], c['description'])
    display(pd.DataFrame([dict(state='eligible', offeringId=x) for x in c['expected']['eligibleOfferingIds']] + [dict(state='excluded', **x) for x in c['expected']['excludedOfferings']] + [dict(state='needs_confirmation', **x) for x in c['expected']['needsConfirmationOfferings']]))
    print('pool:', c['expected']['pool'], 'assertions:', c['expected']['assertions'], 'details:', c['expected']['details'])
print('Cross-case comparisons:', suite['comparisons'])


5. Lịch, ngoại lệ và nguồn hết hạn

Các giờ UTC được đổi sang Asia/Ho_Chi_Minh để xem, không dùng giờ máy hiện tại. [start,end): đúng mở nhận, đúng đóng loại; ngày đóng cửa chặn cả ca hôm trước kéo qua. validationAt dùng kiểm cấu trúc, evaluatedAt dùng cho expected của ca — không che nguồn hết hạn bằng hạn dataset.

In [ ]:
night = datasets['night']
display(pd.DataFrame(night['entities']['weeklySchedules'])[['ownerType','dayOfWeek','startTime','endTime','endDayOffset','is24Hours','timezone','status']])
night_cases = pd.DataFrame([dict(id=c['id'], utc=c['input']['context']['desiredAt'], eligible=c['expected']['eligibleOfferingIds']) for c in suite['cases'] if c['group']=='overnight'])
night_cases['local'] = pd.to_datetime(night_cases['utc'], utc=True).dt.tz_convert('Asia/Ho_Chi_Minh')
display(night_cases)
display(pd.DataFrame(datasets['night-closed-next-day']['entities']['dateExceptions']))
display(pd.DataFrame(datasets['source-expiry']['entities']['dataSources'])[['locator','checkedAt','validUntil','reviewStatus']])


6. Kiểm contract và tính nhất quán của fixtures

Validator chỉ kiểm dữ liệu/expected/coverage, không thực thi GM-30. Giá missing/null, quyền/reviewer mô phỏng và fixtureOnly không được trở thành dữ liệu published. Các nhãn reason chỉ là nhãn fixture, không là enum API app.

In [ ]:
errors = ff.validate_suite(datasets, suite, registry)
assert not errors, errors
assert all(b['fixtureOnly'] for b in datasets.values())
assert {ff.fixture_id('dishes', k) for k,_,_ in ff.DISH_DEFINITIONS} == {r['id'] for r in base['entities']['dishes']}
assert all(c['expected']['pool']['maxSize'] <= 8 for c in suite['cases'])
print({'valid': True, 'groups': len(ff.GROUPS), 'assertions': len(ff.ASSERTIONS), 'eligibilityExecuted': False})
import copy
invalid = copy.deepcopy(suite)
invalid['cases'][0]['expected']['pool']['maxSize'] = 9
display(pd.DataFrame(ff.validate_suite(datasets, invalid, registry)))


7. Xuất, đọc lại và checksum

Tự tạo tests/fixtures/food-data-v1; không ghi vào snapshots khảo sát hoặc seed. Notebook executed nằm datasets Git-ignored. Chạy lại không append, không network/API; chưa kiểm trực tiếp Jupyter kernel.

In [ ]:
stats = ff.export(datasets, suite, registry, OUTPUT_DIR)
assert ff.dc.read_csv_bundle(OUTPUT_DIR / 'base-csv') == base
assert json.loads((OUTPUT_DIR / 'base.json').read_text()) == base
assert json.loads((OUTPUT_DIR / 'datasets.json').read_text()) == datasets
assert json.loads((OUTPUT_DIR / 'cases.json').read_text()) == suite
for path in (OUTPUT_DIR / 'base-csv').glob('*.csv'):
    assert path.read_bytes().startswith(b'\xef\xbb\xbf')
print(stats)
print(OUTPUT_DIR)
display(pd.DataFrame.from_dict(json.loads((OUTPUT_DIR / 'manifest.json').read_text())['files'], orient='index'))
